In [1]:
pip install vosk noisereduce jiwer scipy numpy requests zipfile36

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import urllib.request
import zipfile

# Defining model URLs 
VOSK_MODELS = {
    "en-US": "https://alphacephei.com/vosk/models/vosk-model-small-en-us-0.15.zip",
    "it-IT": "https://alphacephei.com/vosk/models/vosk-model-small-it-0.22.zip",
    "es-ES": "https://alphacephei.com/vosk/models/vosk-model-small-es-0.42.zip"
}

def download_and_extract(lang, url):
    folder_name = f"model-{lang}"
    if os.path.exists(folder_name):
        print(f" Model for {lang} already exists.")
        return folder_name
    
    print(f" Downloading {lang} model... (this may take a moment)")
    file_name = f"{folder_name}.zip"
    urllib.request.urlretrieve(url, file_name)
    
    print(f" Extracting {lang} model...")
    with zipfile.ZipFile(file_name, 'r') as zip_ref:
        zip_ref.extractall(".")
    
    # Renaming extracted folder to standard name
    extracted_name = zip_ref.namelist()[0].split('/')[0]
    os.rename(extracted_name, folder_name)
    os.remove(file_name) # Cleanup zip
    print(f" Ready: {folder_name}")
    return folder_name

# Downloading all 3 models 
for lang, url in VOSK_MODELS.items():
    download_and_extract(lang, url)

 Model for en-US already exists.
 Extracting it-IT model...
 Ready: model-it-IT
 Extracting es-ES model...
 Ready: model-es-ES


In [7]:
import os
import wave
import json
import numpy as np
import noisereduce as nr
from scipy.io import wavfile
from vosk import Model, KaldiRecognizer

# Configuration of models
MODEL_PATHS = {
    "en-US": "model-en-US",
    "it-IT": "model-it-IT",
    "es-ES": "model-es-ES"
}

def recognize_vosk(folder, filename, lang_code):
    """
    1. Loads Audio
    2. Removes Background Noise (Requirement 5)
    3. Transcribes using Vosk (Offline - Requirement 2)
    """
    # 1. Loading the Model
    model_path = MODEL_PATHS.get(lang_code)
    if not os.path.exists(model_path):
        return "[Model Missing]"
    
    model = Model(model_path)

    # 2. Loading Audio File
    full_path = os.path.join(folder, filename)
    if not os.path.exists(full_path):
        return "[File Missing]"
    
    try:
        rate, data = wavfile.read(full_path)
    except ValueError:
        return "[Format Error]"

    # 3. Converting Stereo to Mono
    if len(data.shape) > 1:
        data = data[:, 0]

    # 4. NOISE REDUCTION 
    try:
        clean_data = nr.reduce_noise(y=data, sr=rate, stationary=True, prop_decrease=0.6)
    except:
        clean_data = data 

    # 5. Saving a Temporary File 
    temp_wav = "temp_vosk.wav"
    wavfile.write(temp_wav, rate, clean_data.astype(np.int16))

    # 6. Transcribing
    wf = wave.open(temp_wav, "rb")
    if wf.getnchannels() != 1 or wf.getsampwidth() != 2 or wf.getcomptype() != "NONE":
        return "[Audio Format Error]"

    rec = KaldiRecognizer(model, wf.getframerate())
    rec.SetWords(True)

    results = []
    while True:
        data = wf.readframes(4000)
        if len(data) == 0:
            break
        if rec.AcceptWaveform(data):
            part = json.loads(rec.Result())
            results.append(part.get("text", ""))
    
    final_part = json.loads(rec.FinalResult())
    results.append(final_part.get("text", ""))
    
    wf.close()
    return " ".join([r for r in results if r])

/Users/macbookpro/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
from jiwer import wer

# DATASET CONFIGURATION 
dataset = [
    # ENGLISH (Folder: EN)
    ("EN", "checkin.wav", "en-US", "where is the check-in desk"),
    ("EN", "parents.wav", "en-US", "i have lost my parents"),
    ("EN", "suitcase.wav", "en-US", "please i have lost my suitcase"),
    ("EN", "what_time.wav", "en-US", "what time is my plane"),
    ("EN", "where.wav", "en-US", "where are the restaurants and shops"),

    # MY PERSONAL RECORDINGS 
    (".", "your_sentence1.wav", "en-US", "the flight is delayed"), 
    (".", "your_sentence2.wav", "en-US", "where is the exit"),
    
    # ITALIAN (Folder: IT)
    ("IT", "checkin_it.wav", "it-IT", "dove e il bancone"),
    ("IT", "parents_it.wav", "it-IT", "ho perso i miei genitori"),
    ("IT", "suitcase_it.wav", "it-IT", "per favore ho perso la mia valigia"),
    
    # SPANISH (Folder: ES)
    ("ES", "checkin_es.wav", "es-ES", "dónde están los mostradores"),
    ("ES", "parents_es.wav", "es-ES", "he perdido a mis padres"),
    ("ES", "suitcase_es.wav", "es-ES", "por favor he perdido mi maleta"),
]

#  Running Evaluation 
print(f"{'LANG':<6} | {'FILE':<25} | {'WER':<6} | {'RECOGNIZED TEXT'}")
print("-" * 100)

total_wer = 0
count = 0

for folder, filename, lang, ground_truth in dataset:
    # 1. Transcribing
    hypothesis = recognize_vosk(folder, filename, lang)
    
    # 2. Normalizing the Text
    gt_norm = ground_truth.lower().replace("?", "").replace(".", "").replace("¿", "")
    hyp_norm = hypothesis.lower().replace("?", "").replace(".", "").replace("¿", "")
    
    # 3. Calculating the WER
    if hypothesis.startswith("["): 
        score = 1.0 
    else:
        score = wer(gt_norm, hyp_norm)
    
    total_wer += score
    count += 1
    
    print(f"{lang:<6} | {filename:<25} | {score:.0%} | {hyp_norm}")

# Final Score
if count > 0:
    avg_wer = total_wer / count
    print("-" * 100)
    print(f"Average System WER: {avg_wer:.1%}")
    
    if avg_wer < 0.25:
        print(" PASS: System meets accuracy requirements.")
    else:
        print(" STATUS: OPTIMIZATION NEEDED (Check personal recording text)")

LANG   | FILE                      | WER    | RECOGNIZED TEXT
----------------------------------------------------------------------------------------------------


LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=10 max-active=3000 lattice-beam=2
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:6:7:8:9:10
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 0 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 0 orphan components.
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from model-en-US/ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:282) Loading HCL and G from model-en-US/graph/HCLr.fst model-en-US/graph/Gr.fst
LOG (VoskAPI:ReadDataFiles():model.cc:303) Loading winfo model-en-US/graph/phones/word_boundary.int


en-US  | checkin.wav               | 40% | where is the check in desk


LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=10 max-active=3000 lattice-beam=2
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:6:7:8:9:10
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 0 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 0 orphan components.
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from model-en-US/ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:282) Loading HCL and G from model-en-US/graph/HCLr.fst model-en-US/graph/Gr.fst
LOG (VoskAPI:ReadDataFiles():model.cc:303) Loading winfo model-en-US/graph/phones/word_boundary.int


en-US  | parents.wav               | 20% | i lost my parents


LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=10 max-active=3000 lattice-beam=2
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:6:7:8:9:10
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 0 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 0 orphan components.
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from model-en-US/ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:282) Loading HCL and G from model-en-US/graph/HCLr.fst model-en-US/graph/Gr.fst
LOG (VoskAPI:ReadDataFiles():model.cc:303) Loading winfo model-en-US/graph/phones/word_boundary.int


... (17 outputs left out)

LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=11 max-active=4000 lattice-beam=4
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:6:7:8:9:10
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 0 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 0 orphan components.
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from model-es-ES/ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:282) Loading HCL and G from model-es-ES/graph/HCLr.fst model-es-ES/graph/Gr.fst
LOG (VoskAPI:ReadDataFiles():model.cc:303) Loading winfo model-es-ES/graph/phones/word_boundary.int


es-ES  | parents_es.wav            | 0% | he perdido a mis padres


LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=11 max-active=4000 lattice-beam=4
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:6:7:8:9:10
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 0 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 0 orphan components.
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from model-es-ES/ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:282) Loading HCL and G from model-es-ES/graph/HCLr.fst model-es-ES/graph/Gr.fst
LOG (VoskAPI:ReadDataFiles():model.cc:303) Loading winfo model-es-ES/graph/phones/word_boundary.int


es-ES  | suitcase_es.wav           | 0% | por favor he perdido mi maleta
----------------------------------------------------------------------------------------------------
Average System WER: 9.7%
 PASS: System meets accuracy requirements.
